# 🚢 项目 01：数据清洗 & EDA — Titanic 数据集

> 一条完整的「脏数据 → 清洗 → 探索 → 干净数据」流水线

## 流程
1. 加载数据 + 初览
2. 统计摘要 + 缺失值分析
3. 可视化探索（分布、相关、对比）
4. 数据清洗（缺失值、异常值、编码）
5. 导出干净数据 + EDA 小结

## 0. 导入库

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体 & 样式
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ 环境就绪')
print(f'  numpy: {np.__version__}')
print(f'  pandas: {pd.__version__}')

## 1. 加载数据

In [ ]:
# Titanic 训练集——从本地 data/ 目录加载
URL = '../data/titanic.csv'
df = pd.read_csv(URL)

print(f'📊 数据维度: {df.shape[0]} 行 × {df.shape[1]} 列')
df.head(10)

In [ ]:
# 列的含义速查
info = {
    'PassengerId': '乘客ID',
    'Survived': '是否生还 (0=死亡, 1=生还)',
    'Pclass': '船舱等级 (1/2/3)',
    'Name': '姓名',
    'Sex': '性别',
    'Age': '年龄',
    'SibSp': '同行兄弟姐妹/配偶数',
    'Parch': '同行父母/子女数',
    'Ticket': '船票号码',
    'Fare': '票价',
    'Cabin': '客舱号',
    'Embarked': '登船港口 (C=Cherbourg, Q=Queenstown, S=Southampton)'
}
pd.DataFrame(info.items(), columns=['列名', '含义'])

## 2. 初览：数据类型、缺失值、统计摘要

In [ ]:
# 数据类型 & 非空计数
df.info()

In [ ]:
# 缺失值统计
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'缺失数': missing, '缺失率(%)': missing_pct})
missing_df[missing_df['缺失数'] > 0].sort_values('缺失数', ascending=False)

In [ ]:
# 数值列统计摘要
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# 类别列统计
for col in ['Survived', 'Pclass', 'Sex', 'Embarked']:
    print(f'--- {col} ---')
    print(df[col].value_counts(dropna=False))
    print()

## 3. 可视化探索

> 目标：发现数据中的模式和问题

### 3.1 生还率总览

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 生还比例
surv_counts = df['Survived'].value_counts()
axes[0].pie(surv_counts, labels=['死亡', '生还'], autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[0].set_title('生还率总览')

# 性别 vs 生还
surv_sex = df.groupby('Sex')['Survived'].mean()
surv_sex.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'])
axes[1].set_title('各性别生还率')
axes[1].set_ylabel('生还率')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

# 舱位等级 vs 生还
surv_pclass = df.groupby('Pclass')['Survived'].mean()
surv_pclass.plot(kind='bar', ax=axes[2], color=['#f39c12', '#9b59b6', '#1abc9c'])
axes[2].set_title('各舱位等级生还率')
axes[2].set_ylabel('生还率')
axes[2].set_xticklabels(['1等', '2等', '3等'], rotation=0)

plt.tight_layout()
plt.show()

### 3.2 数值特征分布

In [ ]:
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df[col].hist(bins=40, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].axvline(df[col].median(), color='red', linestyle='--', label=f'中位数={df[col].median():.1f}')
    axes[i].set_title(f'{col} 分布')
    axes[i].legend()

plt.tight_layout()
plt.show()

### 3.3 相关性热力图

In [ ]:
# 数值列相关性
corr_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            linewidths=0.5, square=True)
plt.title('特征相关性热力图')
plt.show()

### 3.4 多维度交叉：性别 × 舱位 × 生还

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
cross = df.pivot_table(index='Sex', columns='Pclass', values='Survived', aggfunc='mean')
sns.heatmap(cross, annot=True, fmt='.2f', cmap='YlGn', ax=ax)
ax.set_title('生还率：性别 × 舱位等级')
plt.show()

### 3.5 年龄分布 × 生还

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for survived, label, color in [(0, '死亡', '#e74c3c'), (1, '生还', '#2ecc71')]:
    subset = df[df['Survived'] == survived]['Age'].dropna()
    ax.hist(subset, bins=30, alpha=0.6, label=label, color=color)
ax.set_xlabel('年龄')
ax.set_ylabel('人数')
ax.set_title('年龄分布：生还 vs 死亡')
ax.legend()
plt.show()

### 3.6 登船港口 × 生还

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

embark_counts = df['Embarked'].value_counts()
embark_counts.plot(kind='bar', ax=ax[0], color=['#3498db', '#e67e22', '#2ecc71'])
ax[0].set_title('各港口登船人数')
ax[0].set_xlabel('港口')

embark_surv = df.groupby('Embarked')['Survived'].mean()
embark_surv.plot(kind='bar', ax=ax[1], color=['#3498db', '#e67e22', '#2ecc71'])
ax[1].set_title('各港口生还率')
ax[1].set_ylabel('生还率')

plt.tight_layout()
plt.show()

## 4. 数据清洗

> 基于探索发现，制定并执行清洗策略

### 4.1 清洗前记录

In [ ]:
print('清洗前:')
print(f'  总行数: {len(df)}')
print(f'  总列数: {len(df.columns)}')
print(f'  缺失值总数: {df.isna().sum().sum()}')
print(f'  重复行: {df.duplicated().sum()}')

### 4.2 删除无用列

`PassengerId` 是索引，`Ticket` 和 `Name` 在基础分析中不直接使用。

In [ ]:
df_clean = df.drop(columns=['PassengerId', 'Ticket', 'Name'])
print(f'已删除: PassengerId, Ticket, Name')
print(f'剩余列: {list(df_clean.columns)}')

### 4.3 处理缺失值

| 列 | 缺失率 | 策略 | 理由 |
|----|--------|------|------|
| Age | ~20% | 用中位数填充 | 分布右偏，中位数更稳健 |
| Cabin | ~77% | 删除该列 | 缺失太多，无法有效利用 |
| Embarked | <1% | 用众数填充 | 缺失极少，模式填充即可 |

In [ ]:
# Cabin: 缺失率 77%，直接删除
df_clean = df_clean.drop(columns=['Cabin'])
print(f'✅ 已删除 Cabin 列（缺失率 77%）')

# Age: 用中位数填充
age_median = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(age_median)
print(f'✅ Age 缺失值 → 用中位数 {age_median:.1f} 填充')

# Embarked: 用众数填充
embarked_mode = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(embarked_mode)
print(f'✅ Embarked 缺失值 → 用众数 {embarked_mode} 填充')

### 4.4 类别编码

In [ ]:
# Sex: 二值编码
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})
print('✅ Sex: male→0, female→1')

# Embarked: One-Hot
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], prefix='Embarked', drop_first=True)
print('✅ Embarked: One-Hot 编码完成')

### 4.5 检查异常值

In [ ]:
# Fare 的箱线图
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_clean['Fare'].plot(kind='box', ax=axes[0])
axes[0].set_title('Fare 箱线图（清洗后）')
axes[0].set_ylabel('票价')

# 票价分布（对数尺度）
df_clean['Fare'].hist(bins=50, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Fare 分布（对数尺度）')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# 票价为零的乘客在哪？
zero_fare = df_clean[df_clean['Fare'] == 0]
print(f'票价为 0 的乘客数: {len(zero_fare)}')
if len(zero_fare) > 0:
    print(zero_fare[['Pclass', 'Age', 'Sex']])

### 4.6 清洗结果检查

In [ ]:
print('清洗后:')
print(f'  总行数: {len(df_clean)}')
print(f'  总列数: {len(df_clean.columns)}')
print(f'  缺失值总数: {df_clean.isna().sum().sum()}')
print(f'  列: {list(df_clean.columns)}')
print()

df_clean.info()

In [ ]:
# 最终数据预览
df_clean.head(10).style.background_gradient(cmap='Blues')

## 5. 导出干净数据

In [ ]:
OUTPUT_PATH = 'output/titanic_clean.csv'
df_clean.to_csv(OUTPUT_PATH, index=False)

import os
size_kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f'✅ 已导出: {OUTPUT_PATH} ({size_kb:.1f} KB)')
print(f'   行数: {len(df_clean)}, 列数: {len(df_clean.columns)}')

---

## EDA 小结

> 把本次探索填写到 `notes/p01-titanic-eda.md`，并在这里写一个摘要。

**模板：**

### 数据概况
- 891 条乘客记录，原始 12 列，清洗后 10 列
- 生还率：**38.4%**（342/891），整体死亡率高于生还率

### 核心发现
1. **性别是最强预测因子**——女性生还率 74%，男性仅 19%
2. **舱位等级显著影响生还**——1 等舱生还率 63%，3 等舱仅 24%
3. **年龄有一定影响**——儿童生还率较高，20-40 岁男性死亡率最高
4. **登船港口有微弱差异**——Cherbourg 上船的乘客生还率最高（55%）
5. **票价和舱位高度相关**——票价本身可能只是代理变量

### 清洗决策
| 列 | 决策 | 理由 |
|----|------|------|
| Cabin | 删除 | 缺失率 77%，信息量不足 |
| Age | 中位数填充 | 20% 缺失，中位数比均值稳健 |
| Embarked | 众数填充 | <1% 缺失，影响极小 |
| Sex | 二值编码 | 只有 male/female |
| Embarked | One-Hot | 3 类名义变量 |
| Name/Ticket | 删除 | 基础 EDA 不需要 |

### 下一步
数据清洗完成。可以用于：
- 项目 02：手写线性回归预测生还
- 项目 03：sklearn 分类器对比
- 特征工程进阶：从 Name 提取 Title（Mr/Mrs/Miss/Dr）